# Sementic Search and Retrieval Augmentation

Search was one of the first language model applications to see broad industry adoption. The ability they add is called semantic search, which enables searching by meaning, and not simply keyword matching. On a separate track, the fast adoption of text generation models led many users to ask the models questions and expect factual answers. This problem grew to be known as model “hallucinations,” and one of the leading ways to reduce it is to build systems that can retrieve relevant information and provide it to the LLM to aid it in generating more factual answers. This method, called RAG, is one of the most popular applications of LLMs.

## Overview of Semantic Search and RAG

There’s a lot of research on how to best use language models for search. Three broad categories of these models are dense retrieval, reranking, and RAG. 

*Dense Retrieval*

Dense retrieval systems rely on the concept of embeddings, the same concept we’ve encountered in the previous notebooks, and turn the search problem into retrieving the nearest neighbors of the search query (after both the query and the documents are converted into embeddings).

![Alt Text](./figures/dense-retrieval.png)

*Reranking*

Search systems are often pipelines of multiple steps. A reranking language model is one of these steps and is tasked with scoring the relevance of a subset of results against the query; the order of results is then changed based on these scores.

![Alt Text](./figures/rerank.png)

*RAG*

The growing LLM capability of text generation led to a new type of search systems that include a model that generates an answer in response to a query. Generative search is a subset of a broader type of category of systems better called RAG systems. These are text generation systems that incorporate search capabilities to reduce hallucinations, increase factuality, and/or ground the generation model on a specific dataset. RAG system formulates an answer to a question and (preferably) cites its information sources. 

![Alt Text](./figures/RAG.png)

## Semantic Search with Language Models

### Dense Retrieval 

Recall that embeddings turn text into numeric representations. Those can be thought of as points in space, as we can see below. Points that are close together mean that the text they represent is similar. So in this example, text 1 and text 2 are more similar to each other (because they are near each other) than text 3 (because it’s farther away).

![Alt Text](./figures/dense-retr-1.png)

This is the property that is used to build search systems. In this scenario, when a user enters a search query, we embed the query, thus projecting it into the same space as our text archive. Then we simply find the nearest documents to the query in that space, and those would be the search results

![Alt Text](./figures/dense-retr-2.png)

Judging by the distances in the figure above, “text 2” is the best result for this query, followed by “text 1.” Two questions could arise here, however:

* Should text 3 even be returned as a result? That’s a decision for you, the system designer. It’s sometimes desirable to have a max threshold of similarity score to filter out irrelevant results (in case the corpus has no relevant results for the query).
* Are a query and its best result semantically similar? Not always. This is why language models need to be trained on question-answer pairs to become better at retrieval. This process is explained in more detail in Chapter 10.
  
Below shows how we chunk a document before proceeding to embed each chunk. Those embedding vectors are then stored in the vector database and are ready for retrieval.

![Alt text](./figures/dense-retr-3.png)

#### Dense Retrieval Example 

Let’s take a look at a dense retrieval example by using Cohere to search the Wikipedia page for the film Interstellar. In this example, we will do the following:

1. Get the text we want to make searchable and apply some light processing to chunk it into sentences.
2. Embed the sentences.
3. Build the search index.
4. Search and see the results.

Note: you must get a Cohere API key by signing up at [Cohere](https://cohere.com)

In [ ]:
import cohere
import numpy as np
import pandas as pd
from tqdm import tqdm 

api_key = 'my-key'

# Connect to Cohere client
co = cohere.Client(api_key)

In [12]:
# Getting the text archive and chunking it 
# We'll use the first section of the Wikipedia article on the film Interstellar. 
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan. 
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine. 
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007. 
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar. 
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm. 
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles. 
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles. 
In the United States, it was first released on film stock, expanding to venues using digital projectors. 
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014. 
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight. 
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades
"""

# Split into a list of sentences 
texts = text.split('.')

# Clean up to remove empty spaces and new lines
texts = [t.strip(' \n') for t in texts]

In [9]:
# Embedding the text chunks
# We'll send them to the Cohere API and get back a vector for each text 
response = co.embed(
    texts=texts, 
    model="embed-english-v3.0",
    input_type="search_document", 
).embeddings

embeds = np.array(response)
print(embeds.shape)
print("We have 15 vectors, each one of size 1024")

(15, 1024)
We have 15 vectors, each one of size 1024


In [14]:
# Building the search index 
# An index stores the embeddings and is optimized to quickly retrieve the nearest neighbors even if we have very large number of points
import faiss # open source library for efficient similarity search and clustering of dense vectors

dim = embeds.shape[1] # num of features per vector
index = faiss.IndexFlatL2(dim) # build the index, perfoms brute force to find the nearest neighbors using the Euclidean distance
print(index.is_trained)
index.add(np.float32(embeds)) # add vectors to the index

True


In [15]:
# Search the index 
def search(query, number_of_results=3): 
    # 1. Get the query's embedding
    query_embed = co.embed(texts=[query], 
                    model="embed-english-v3.0",
                    input_type="search_query",).embeddings[0]

    # 2. Retrieve the nearest neighbors
    distances , similar_item_ids = index.search(np.float32([query_embed]), number_of_results) 

    # 3. Format the results
    texts_np = np.array(texts) # Convert texts list to numpy for easier indexing
    results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]], 
                                'distance': distances[0]})
    
    # 4. Print and return the results
    print(f"Query:'{query}'\nNearest neighbors:")
    return results

In [16]:
# We are now ready to write a query and search the texts!
query = "How precise was the science"
results = search(query)
results

Query:'How precise was the science'
Nearest neighbors:


,texts,distance
0,It has also received praise from many astronom...,1.200760
1,Caltech theoretical physicist and 2017 Nobel l...,1.370212
2,Interstellar uses extensive practical and mini...,1.580641


In [20]:
# Would this be possible if we were only doing keyword search because the top result did not include the same keywords in the query? 
# We'll verify by using the BM25 algorithm - the leading lexical search methods. 
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string 

def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc


tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

def keyword_search(query, top_k=3, num_candidates=15):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
    
    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

100%|██████████| 15/15 [00:00<00:00, 76075.65it/s]


In [21]:
keyword_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine


Notice that we get different results from the dense retrieval search. Note that the first result does not really answer the question despite it sharing the word “science” with the query. In the next section, we’ll see how adding a reranker can improve this search system. 

#### Caveats of dense retrieval

It’s useful to be aware of some of the drawbacks of dense retrieval and how to address them. What happens, for example, if the texts don’t contain the answer? We still get results and their distances. For example:

In [22]:
query = "What is the mass of the moon?"
results = search(query)
results

Query:'What is the mass of the moon?'
Nearest neighbors:


,texts,distance
0,It has also received praise from many astronom...,1.468251
1,Caltech theoretical physicist and 2017 Nobel l...,1.549838
2,"Since its premiere, Interstellar gained a cult...",1.550955


We get irrelevant results. In cases like this, one possible heuristic is to set a threshold level—a maximum distance for relevance, for example. A lot of search systems present the user with the best info they can get and leave it up to the user to decide if it’s relevant or not. Tracking the information of whether the user clicked on a result (and were satisfied by it) can improve future versions of the search system.

Another caveat of dense retrieval is when a user wants to find an exact match for a specific phrase. That’s a case that’s perfect for keyword matching. That’s one reason why hybrid search, which includes both semantic search and keyword search, is advised instead of relying solely on dense retrieval.

Dense retrieval systems also find it challenging to work properly in domains other than the ones that they were trained on. So, for example, if you train a retrieval model on internet and Wikipedia data, and then deploy it on legal texts (without having enough legal data as part of the training set), the model will not work as well in that legal domain.

The final thing we’d like to point out is that this is a case where each sentence contained a piece of information, and we showed queries that specifically ask for that information. What about questions whose answers span multiple sentences? This highlights one of the important design parameters of dense retrieval systems: what is the best way to chunk long texts? And why do we need to chunk them in the first place?

#### Chunking Long Texts

One limitation of Transformer language models is that they are limited in context sizes, meaning we cannot feed them very long texts that go above the number of words or tokens that the model supports. So how do we embed long texts?

There are several possible ways, and two possible approaches shown below include indexing one vector per document and indexing multiple vectors per document.

![Alt text](./figures/dense-retr-4.png)

##### One Vector per Document 

In this approach, we use a single vector to represent the whole document. The possibilities here include:

* Embedding only a representative part of the document and ignoring the rest of the text. This may mean embedding only the title, or only the beginning of the document. This is useful to get quickly started with building a demo but it leaves a lot of information unindexed and therefore unsearchable. As an approach, it may work better for documents where the beginning captures the main points of a document (think: Wikipedia article). But it’s really not the best approach for a real system because a lot of information would be left out of the index and would be unsearchable.
* Embedding the document in chunks, embedding those chunks, and then aggregating those chunks into a single vector. The usual method of aggregation here is to average those vectors. A downside of this approach is that it results in a highly compressed vector that loses a lot of the information in the document.

This approach can satisfy some information needs, but not others. A lot of the time, a search is for a specific piece of information contained in an article, which is better captured if the concept had its own vector.

##### Multipe Vectors per Document

In this approach, we chunk the document into smaller pieces, and embed those chunks. Our search index then becomes that of chunk embeddings, not entire document embeddings. Below shows a number of possible text chunking approaches.

![Alt text](./figures/dense-retr-5.png)

The chunking approach is better because it has full coverage of the text and because the vectors tend to capture individual concepts inside the text. This leads to a more expressive search index. Below shows a number of possible approaches.

![Alt text](./figures/dense-retr-6.png)

The best way of chunking a long text will depend on the types of texts and queries your system anticipates. Approaches include:

* Each sentence is a chunk. The issue here is this could be too granular and the vectors don’t capture enough of the context.
* Each paragraph is a chunk. This is great if the text is made up of short paragraphs. Otherwise, it may be that every 3–8 sentences is a chunk.
* Some chunks derive a lot of their meaning from the text around them. So we can incorporate some context via:
    * Adding the title of the document to the chunk.
    * Adding some of the text before and after them to the chunk. This way, the chunks can overlap so they include some surrounding text that also appears in adjacent chunks. This is what we can see in the third example above.
  
Expect more chunking strategies to arise as the field develops—some of which may even use LLMs to dynamically split a text into meaningful chunks.

#### Nearest Neighbor Search Versus Vector Databases

Once the query is embedded, we need to find the nearest vectors to it from our text archive as we can see below. The most straightforward way to find the nearest neighbors is to calculate the distances between the query and the archive. That can easily be done with NumPy and is a reasonable approach if you have thousands or tens of thousands of vectors in your archive.

![Alt text](./figures/dense-retr-7.png)

As you scale beyond to the millions of vectors, an optimized approach for retrieval is to rely on approximate nearest neighbor search libraries like `Annoy` or `FAISS`. These allow you to retrieve results from massive indexes in milliseconds and some of them can improve their performance by utilizing GPUs and scaling to clusters of machines to serve very large indices.

Another class of vector retrieval systems are vector databases like `Weaviate` or `Pinecone`. A vector database allows you to add or delete vectors without having to rebuild the index. They also provide ways to filter your search or customize it in ways beyond merely vector distances.

#### Fine-tuning Embedding Models for Dense Retrieval

Just as we discussed in notebook 4 on text classification, we can improve the performance of an LLM on a task using fine-tuning. As in that case, retrieval needs to optimize text embeddings and not simply token embeddings. The process for this fine-tuning is to get training data composed of queries and relevant results.

Let’s look at one example from our dataset, the sentence “Interstellar premiered on October 26, 2014, in Los Angeles.” Two possible queries where this is a relevant result are:

* Relevant query 1: “Interstellar release date”
* Relevant query 2: “When did Interstellar premier”

The fine-tuning process aims to make the embeddings of these queries close to the embedding of the resulting sentence. It also needs to see negative examples of queries that are not relevant to the sentence, for example:

* Irrelevant query: “Interstellar cast”

With these examples, we now have three pairs—two positive pairs and one negative pair. Let’s assume, as we can see below, that before fine-tuning, all three queries have the same distance from the result document. That’s not far-fetched because they all talk about Interstellar.

![Alt text](./figures/dense-retr-8.png)

The fine-tuning step works to make the relevant queries closer to the document and at the same time make irrelevant queries farther from the document. 

![Alt text](./figures/dense-retr-9.png)

### Reranking

Rerank is tasked with changing the order of the search results based on relevance to the search query. This one step can vastly improve search results and it’s in fact what Microsoft Bing added to achieve the improvements to search results using BERT-like models. Below shows the structure of a rerank search system serving as the second stage in a two-stage search system.

![Alt text](./figures/rerank-system.png)

#### Reranking Example 

A reranker takes in the search query and a number of search results, and returns the optimal ordering of these documents so the most relevant ones to the query are higher in ranking. Cohere’s Rerank endpoint is a simple way to start using a first reranker. 

In [24]:
query = "how precise was the science?"
results = co.rerank(query=query, documents=texts, top_n=3, return_documents=True)

for idx, result in enumerate(results.results): 
    print(idx, result.relevance_score, result.document.text)

0 0.13634396 It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
1 0.038053043 The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
2 0.033453528 Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time


This shows the reranker is much more confident about the first result, assigning it a relevance score of 0.16, while the other results are scored much lower in relevance.

In this basic example, we passed our reranker all 15 of our documents. More often, however, our index would have thousands or millions of entries, and we need to shortlist, say one hundred or one thousand results and then present those to the reranker. This shortlisting step is called the *first stage* of the search pipeline.

The first-stage retriever can be keyword search, dense retrieval, or better yet—hybrid search that uses both of them. We can revisit our previous example to see how adding a reranker after a keyword search system improves its performance.

Let’s tweak our keyword search function so it retrieves a list of the top 10 results using keyword search, then use rerank to choose the top 3 results from those 10:

In [25]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
    
    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

   
    #Add re-ranking
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]
    
    print(f"\nTop-3 hits by rank-API ({len(bm25_hits)} BM25 hits re-ranked)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)
    # print(results.results)
    for hit in results.results:
        # print(hit)
        print("\t{:.3f}\t{}".format(hit.relevance_score, hit.document.text.replace("\n", " ")))

Now we can send our query and check the results of keyword search and then the result of keyword search shortlisting its top 10 results, then pass them on to the reranker:

In [27]:
keyword_and_reranking_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects

Top-3 hits by rank-API (10 BM25 hits re-ranked)
	0.035	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	0.032	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine
	0.031	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of In

This is a toy example that gives us a glimpse of the effect, but in practice, such a pipeline significantly improves search quality. On a multilingual benchmark like MIRACL, a reranker can boost performance from 36.5 to 62.8, measured as nDCG@10 (more on evaluation later in this chapter).

#### Open source retrieval and reranking with sentence transformers

To locally set up retrieval and reranking on your own machine, you can use the Sentence Transformers library. Refer to the documentation at [SBERT.net](https://www.sbert.net) for setup. Check the [“Retrieve & Re-Rank” section](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html) for instructions and code examples for how to conduct these steps in the library.

#### How Reranking Models Work 

One popular way of building LLM search rerankers is to present the query and each result to an LLM working as a *cross-encoder*. This means that a query and possible result are presented to the model at the same time allowing the model to view both these texts before it assigns a relevance score, as we can see in below. All of the documents are processed simultaneously as a batch yet each document is evaluated against the query independently. The scores then determine the new order of the results. This method is described in more detail in a paper titled “Multi-stage document ranking with BERT” and is sometimes referred to as monoBERT.

![alt text](./figures/reranking-cross-encoder.png)

This formulation of search as relevance scoring basically boils down to being a classification problem. Given those inputs, the model outputs a score from 0–1 where 0 is irrelevant and 1 is highly relevant. This should be familiar from our classification discussions in notebook 4.

### Retrieval Evaluation Metrics

Semantic search is evaluated using metrics from the Information Retrieval (IR) field. Let’s discuss one of these popular metrics: mean average precision (MAP). Evaluating search systems needs three major components: a text archive, a set of queries, and relevance judgments indicating which documents are relevant for each query. 

![alt text](./figures/retrieval-eval-met.png)

Using this test suite, we can proceed to explore evaluating search systems. Let’s start with a simple example. Let’s assume we pass query 1 to two different search systems. And get two sets of results. Say we limit the number of results to three, 

![alt text](./figures/retrieval_eval-met-2.png)

To tell which is a better system, we turn to the relevance judgments that we have for the query. 

![alt text](./figures/retrieval-eval-met-3.png)

This shows us a clear case where system 1 is better than system 2. Intuitively, we may just count how many relevant results each system retrieved. System 1 got two out of three correct, and system 2 got only one out of three correct. But what about a case like below where both systems only get one relevant result out of three, but they’re in different positions?

![alt text](./figures/retrieval-eval-met-4.png)

In this case, we can intuit that system 1 did a better job than system 2 because the result in the first position (the most important position) is correct. But how can we assign a number or score to how much better that result is? Mean average precision is a measure that is able to quantify this distinction.

#### Scoring a single query with Average Precision

To score a search system on this query, we can focus on scoring the relevant documents. Let’s start by looking at a query that only has one relevant document in the test suite. 

The first one is easy: the search system placed the relevant result (the only available one for this query) at the top. This gets the system the perfect score of 1. Below shows this calculation: looking at the first position, we have a relevant result leading to a precision at position 1 of 1.0 (calculated as the number of relevant results at position 1, divided by the position we’re currently looking at).

![alt text](./figures/retrieval-eval-met-5.png)

Since we’re only scoring relevant documents we can ignore the scores of nonrelevant documents and stop our calculation here. What if the system actually placed the only relevant result at the third position, however? How would that affect the score? Below shows how that results in a penalty.

![alt text](./figures/retrieval-eval-met-6.png)

Let’s now look at a query with more than one relevant document. Below shows that calculation and how averaging now comes into the picture.

![alt text](./figures/retrieval-eval-met-7.png)

#### Scoring across multiple queries with Mean Average Precision

Now that we’re familiar with precision at k and average precision, we can extend this knowledge to a metric that can score a search system against all the queries in our test suite. That metric is called mean average precision.

![alt text](./figures/retrieval-eval-met-8.png)

Now we have a single metric that we can use to compare different systems. If you want to learn more about evaluation metrics, see the “Evaluation in Information Retrieval” chapter of Introduction to Information Retrieval (Cambridge University Press) by Christopher D. Manning, Prabhakar Raghavan, and Hinrich Schütze.

In addition to mean average precision, another metric commonly used for search systems is normalized discounted cumulative gain (nDCG), which is more nuanced in that the relevance of documents is not binary (relevant versus not relevant) and one document can be labeled as more relevant than another in the test suite and scoring mechanism.

## Retrieval-Augmented Generation (RAG)

The mass adoption of LLMs quickly led to people asking them questions and expecting factual answers. While the models can answer some questions correctly, they also confidently answer lots of questions incorrectly. The leading method the industry turned to remedy this behavior is RAG, described in the paper “Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks”, 

![alt text](./figures/RAG-2.png)

RAG systems incorporate search capabilities in addition to generation capabilities.

## From Search to RAG

Let’s now turn our search system into a RAG system. We do that by adding an LLM to the end of the search pipeline. We present the question and the top retrieved documents to the LLM, and ask it to answer the question given the context provided by the search results. 

![alt text](./figures/RAG-3.png)

This generation step is called grounded generation because the retrieved relevant information we provide the LLM establishes a certain context that grounds the LLM in the domain we’re interested in. Below shows how grounded generation fits after search if we continue our embeddings search example from earlier.

![alt text](./figures/RAG-4.png)

Find the most relevant information to an input prompt by comparing the similarities between embeddings. The most relevant information is added to the prompt before giving it to the LLM. 


## Example: Grounded Generation with an LLM API

Let’s look at how to add a grounded generation step after the search results to create our first RAG system. For this example, we’ll use Cohere’s managed LLM, which builds on the search systems we’ve seen earlier in the chapter. We’ll use embedding search to retrieve the top documents, then we’ll pass those to the co.chat endpoint along with the questions to provide a grounded answer:

In [28]:
query = "income generated"

# 1 - Retrieval
# We'll use embedding search from Dense Retrieval. But ideally we'd do hybrid
results = search(query)

# 2 - Grounded Generation
docs_dict = [{'text': text} for text in results['texts']]
response = co.chat(
    message = query,
    documents = docs_dict
)
print(response.text)

Query:'income generated'
Nearest neighbors:
The film generated $677 million worldwide, and $773 million with subsequent re-releases.


## Example: RAG with Local Models

Let us now replicate this basic functionality with local models. We will lose the ability to do span citations and the smaller local model isn’t going to work as well as the larger managed model, but it’s useful to demonstrate the flow. 

### Loading the generation model

In [37]:
# we start by downloading a quantized model
# !wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
# I've already downloaded the model and stored it in the same directory of my notebooks

In [8]:
# using llama.cpp, llama-cpp-python, and langchain, we load the text generation model
from langchain import LlamaCpp

llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048, 
    seed=42,
    verbose=False, 
    stop=["<|end|>"] 
)


### Loading the embedding model

In this example, we will choose the avsolatorio/GIST-small-Embedding-v0 model. At the time of writing, it is high on the [MTEB leaderboard](https://mteb-leaderboard.hf.space) for embedding models and relatively small. This will load a small sentence-embedding model (384 dims) and maps text to vectors so that semantically similar text lands close together.

*Note: kernel may crash b/c of too many models loaded. Restart the kernel and run only this example section if you keep running into crashes.*

In [1]:
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

# Embedding model for converting text to numerical representation
embedding_model = HuggingFaceEmbeddings(
    model_name = "avsolatorio/GIST-small-Embedding-v0"
)

/var/folders/lt/dwt7vtzj1sx2nvf4fgrn8r_00000gn/T/ipykernel_72718/1310304915.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model = HuggingFaceEmbeddings(
/Users/brianmorales/miniconda3/envs/thellmbook/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
W1011 10:23:40.512000 72718 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
/Users/brianmorales/miniconda3/envs/thel

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
# Getting the text archive and chunking it 
# We'll use the first section of the Wikipedia article on the film Interstellar. 
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan. 
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine. 
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007. 
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar. 
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm. 
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles. 
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles. 
In the United States, it was first released on film stock, expanding to venues using digital projectors. 
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014. 
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight. 
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades
"""

# Split into a list of sentences 
texts = text.split('.')

# Clean up to remove empty spaces and new lines
texts = [t.strip(' \n') for t in texts]

FAISS will embed every string in `texts` and indexes the vectors in FAISS as we've done earlier in this chapter. 

In [5]:
# setup our vector database
from langchain.vectorstores import FAISS

# Create the local vector database
db = FAISS.from_texts(texts, embedding_model)

### RAG Prompt

A prompt template plays a vital part in the RAG pipeline. It is the central place where we communicate the relevant documents to the LLM. To do so, we will create an additional input variable named context that can provide the LLM with the retrieved documents:

In [6]:
from langchain import PromptTemplate

# Create a prompt template
template = """<|user|>
Relevant information:
{context}

Provide a concise answer the following question using the relevant information provided above:
{question}<|end|>
<|assistant|>
"""

prompt = PromptTemplate(
    template = template, 
    input_variables=["context", "question"]
)

In [ ]:
from langchain.chains import RetrievalQA

# RAG Pipeline
rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff', # concatenates all retrieved chunks into on {context} block, one LLM call. Other types: map_reduce, ...
    retriever=db.as_retriever(), # top-k similarity search in FAISS (defualt k=4, cosine/L2 depending on index)
    chain_type_kwargs={
        "prompt":prompt
    },
    verbose=True
)

In [10]:
# Now we're ready to call the model
rag.invoke("Income generated")



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Income generated',
 'result': ' Interstellar grossed over $677 million worldwide, with subsequent re-releases bringing its total to approximately $773 million. This made it the tenth-highest grossing film of 2014.'}

## Advanced RAG Techiniqes

### Query Rewriting

If the RAG system is a chatbot, the preceding simple RAG implementation would likely struggle with the search step if a question is too verbose, or to refer to context in previous messages in the conversation. This is why it’s a good idea to use an LLM to rewrite the query into one that aids the retrieval step in getting the right information. An example of this is a message such as:

    User Question: “We have an essay due tomorrow. We have to write about some animal. I love penguins. I could write about them. But I could also write about dolphins. Are they animals? Maybe. Let’s do dolphins. Where do they live for example?”

This should actually be rewritten into a query like:

    Query: “Where do dolphins live”

This rewriting behavior can be done through a prompt (or through an API call). Cohere’s API, for example, has a dedicated query-rewriting mode for `co.chat`.

### Multi-query RAG

The next improvement we can introduce is to extend the query rewriting to be able to search multiple queries if more than one is needed to answer a specific question. Take for example:

    User Question: “Compare the financial results of Nvidia in 2020 vs. 2023”

We may find one document that contains the results for both years, but more likely, we’re better off making two search queries:

    Query 1: “Nvidia 2020 financial results”

    Query 2: “Nvidia 2023 financial results”
    
We then present the top results of both queries to the model for grounded generation. An additional small improvement here is to also give the query rewriter the option to determine if no search is required and if it can directly generate a confident answer without searching.

### Multi-hop RAG
A more advanced question may require a series of sequential queries. Take for example a question like:

    User Question: “Who are the largest car manufacturers in 2023? Do they each make EVs or not?”

To answer this, the system must first search for:

    Step 1, Query 1: “largest car manufacturers 2023”

Then after it gets this information (the result being Toyota, Volkswagen, and Hyundai), it should ask follow-up questions:

    Step 2, Query 1: “Toyota Motor Corporation electric vehicles”

    Step 2, Query 2: “Volkswagen AG electric vehicles”

    Step 2, Query 3: “Hyundai Motor Company electric vehicles”

### Query Routing

An additional enhancement is to give the model the ability to search multiple data sources. We can, for example, specify for the model that if it gets a question about HR, it should search the company’s HR information system (e.g., Notion) but if the question is about customer data, that it should search the customer relationship management (CRM) (e.g., Salesforce).

### Agentic RAG

You may be able to now see that the list of previous enhancements slowly delegates more and more responsibility to the LLM to solve more and more complex problems. This relies on the LLM’s capability to gauge the required information needs as well as its ability to utilize multiple data sources. This new nature of the LLM starts to become closer and closer to an agent that acts on the world. The data sources can also now be abstracted into tools. We saw, for example, that we can search Notion, but by the same token, we should be able to post to Notion as well.

Not all LLMs will have the RAG capabilities mentioned here. At the time of writing, likely only the largest managed models may be able to attempt this behavior. Thankfully, Cohere’s Command R+ excels at these tasks and is available as an open-weights model as well.

## RAG Evaluation

There are still ongoing developments in how to evaluate RAG models. A good paper to read on this topic is “Evaluating verifiability in generative search engines” (2023), which runs human evaluations on different generative search systems.

It evaluates results along four axes:

* Fluency: Whether the generated text is fluent and cohesive.
* Perceived utility: Whether the generated answer is helpful and informative.
* Citation recall: The proportion of generated statements about the external world that are fully supported by their citations.
* Citation precision: The proportion of generated citations that support their associated statements.

While human evaluation is always preferred, there are approaches that attempt to automate these evaluations by having a capable LLM act as a judge (called *LLM-as-a-judge*) and score the different generations along the different axes. `Ragas` is a software library that does exactly this. It also scores some additional useful metrics like:

* Faithfulness: Whether the answer is consistent with the provided context
* Answer relevance: How relevant the answer is to the question
The [Ragas documentation site](https://docs.ragas.io/en/stable/) provides more details about the formulas to actually calculate these metrics.

In the next chapter, we will explore how language models can be made multimodal and reason not just about text but images as well.